# Exercícios — SQL Intermediário com SQLite — Gabarito

Este notebook resolve os mesmos 12 exercícios do notebook do aluno usando a base oficial do minicurso: clientes, pedidos, produtos e itens_pedido.

No Google Colab, a célula inicial baixa utils/gerar_dados.py do repositório e recria minicurso_aula2.db com a semente 42.

In [22]:
import sqlite3
from pathlib import Path
from urllib.request import urlretrieve

ARQUIVO_GERADOR = Path("gerar_dados.py")
ARQUIVO_BANCO = Path("minicurso_aula2.db")
URL_GERADOR = (
    "https://raw.githubusercontent.com/PET-Estat-UFSCar/minicurso-SQL/"
    "main/Aula%202%20-%20Intermedi%C3%A1rio/utils/gerar_dados.py"
)

if not ARQUIVO_GERADOR.exists():
    urlretrieve(URL_GERADOR, ARQUIVO_GERADOR)
namespace = {}
exec(ARQUIVO_GERADOR.read_text(encoding="utf-8"), namespace)
namespace["criar_banco"](ARQUIVO_BANCO)

conexao = sqlite3.connect(ARQUIVO_BANCO)
conexao.execute("PRAGMA foreign_keys = ON")
cursor = conexao.cursor()
print(f"Banco conectado: {ARQUIVO_BANCO.resolve()}")


Banco criado com sucesso!
Arquivo: /home/usuario/Projetos/minicurso-SQL/Aula 2 - Intermediário/Material Completo/Exercícios/minicurso_aula2.db

Resumo da base:
  Clientes: 100
  Pedidos: 250
  Produtos: 40
  Itens de pedido: 864
  Clientes sem pedido: 10
  Produtos sem venda: 5

Banco conectado: /home/usuario/Projetos/minicurso-SQL/Aula 2 - Intermediário/Material Completo/Exercícios/minicurso_aula2.db


## Exercício 1 — Inspecionando o schema

A resolução consulta a estrutura das quatro tabelas e suas chaves estrangeiras.

In [23]:
for tabela in ("clientes", "pedidos", "produtos", "itens_pedido"):
    print(f"\n--- {tabela} ---")
    for linha in cursor.execute(f"PRAGMA table_info({tabela})"):
        print(linha)

for tabela in ("pedidos", "itens_pedido"):
    print(f"\n--- FKs de {tabela} ---")
    for linha in cursor.execute(f"PRAGMA foreign_key_list({tabela})"):
        print(linha)


--- clientes ---
(0, 'id_cliente', 'INTEGER', 0, None, 1)
(1, 'nome', 'TEXT', 1, None, 0)
(2, 'cidade', 'TEXT', 1, None, 0)

--- pedidos ---
(0, 'id_pedido', 'INTEGER', 0, None, 1)
(1, 'id_cliente', 'INTEGER', 1, None, 0)
(2, 'data_pedido', 'TEXT', 1, None, 0)

--- produtos ---
(0, 'id_produto', 'INTEGER', 0, None, 1)
(1, 'nome', 'TEXT', 1, None, 0)
(2, 'categoria', 'TEXT', 1, None, 0)
(3, 'preco', 'REAL', 1, None, 0)

--- itens_pedido ---
(0, 'id_pedido', 'INTEGER', 1, None, 1)
(1, 'id_produto', 'INTEGER', 1, None, 2)
(2, 'quantidade', 'INTEGER', 1, None, 0)

--- FKs de pedidos ---
(0, 0, 'clientes', 'id_cliente', 'id_cliente', 'NO ACTION', 'NO ACTION', 'NONE')

--- FKs de itens_pedido ---
(0, 0, 'produtos', 'id_produto', 'id_produto', 'NO ACTION', 'NO ACTION', 'NONE')
(1, 0, 'pedidos', 'id_pedido', 'id_pedido', 'NO ACTION', 'NO ACTION', 'NONE')


## Exercício 2 — Tamanho e características da base

A consulta confirma os totais conhecidos e usa LEFT JOIN para encontrar ausências.

In [24]:
consultas = {
    "clientes": "SELECT COUNT(*) FROM clientes",
    "pedidos": "SELECT COUNT(*) FROM pedidos",
    "produtos": "SELECT COUNT(*) FROM produtos",
    "itens_pedido": "SELECT COUNT(*) FROM itens_pedido",
    "clientes_sem_pedido": """
        SELECT COUNT(*)
        FROM clientes
        LEFT JOIN pedidos ON pedidos.id_cliente = clientes.id_cliente
        WHERE pedidos.id_pedido IS NULL
    """,
    "produtos_sem_venda": """
        SELECT COUNT(*)
        FROM produtos
        LEFT JOIN itens_pedido ON itens_pedido.id_produto = produtos.id_produto
        WHERE itens_pedido.id_produto IS NULL
    """,
}
for nome, sql in consultas.items():
    print(f"{nome}: {cursor.execute(sql).fetchone()[0]}")

clientes: 100
pedidos: 250
produtos: 40
itens_pedido: 864
clientes_sem_pedido: 10
produtos_sem_venda: 5


## Exercício 3 — INNER JOIN: pedidos e clientes

Uma solução possível:

In [25]:
sql = """
SELECT pedidos.id_pedido, clientes.nome AS cliente, clientes.cidade, pedidos.data_pedido
FROM pedidos
INNER JOIN clientes ON clientes.id_cliente = pedidos.id_cliente
ORDER BY pedidos.id_pedido
LIMIT 15;
"""
cursor.execute(sql)
cursor.fetchall()


[(1, 'Eduarda Ribeiro', 'Piracicaba', '2026-01-15'),
 (2, 'Henrique Mendes', 'Rio Claro', '2026-09-11'),
 (3, 'Yasmin Nascimento', 'Rio Claro', '2026-06-16'),
 (4, 'Yasmin Pereira', 'Limeira', '2026-04-04'),
 (5, 'Paula Santos', 'Sorocaba', '2026-09-07'),
 (6, 'Beatriz Costa', 'Piracicaba', '2026-04-19'),
 (7, 'Gabriel Araújo', 'Limeira', '2026-07-01'),
 (8, 'Kaique Carvalho', 'Rio Claro', '2026-05-13'),
 (9, 'Beatriz Ribeiro', 'Bauru', '2026-06-24'),
 (10, 'Paula Martins', 'Rio Claro', '2026-05-24'),
 (11, 'Elisa Carvalho', 'Piracicaba', '2026-05-22'),
 (12, 'Natalia Costa', 'São Paulo', '2026-01-06'),
 (13, 'Felipe Costa', 'Piracicaba', '2026-09-22'),
 (14, 'Caio Silva', 'Piracicaba', '2026-04-08'),
 (15, 'Larissa Rocha', 'Ribeirão Preto', '2026-02-13')]

## Exercício 4 — LEFT JOIN: clientes sem pedido

Uma solução possível:

In [26]:
sql = """
SELECT clientes.id_cliente, clientes.nome, clientes.cidade
FROM clientes
LEFT JOIN pedidos ON pedidos.id_cliente = clientes.id_cliente
WHERE pedidos.id_pedido IS NULL
ORDER BY clientes.id_cliente;
"""
cursor.execute(sql)
cursor.fetchall()

[(91, 'Otavio Martins', 'Piracicaba'),
 (92, 'Paula Carvalho', 'Sorocaba'),
 (93, 'Yasmin Silva', 'Sorocaba'),
 (94, 'Diego Rodrigues', 'Bauru'),
 (95, 'Beatriz Mendes', 'Ribeirão Preto'),
 (96, 'Beatriz Silva', 'Bauru'),
 (97, 'Julia Monteiro', 'Ribeirão Preto'),
 (98, 'Eduarda Fernandes', 'Campinas'),
 (99, 'Igor Fernandes', 'Piracicaba'),
 (100, 'Caio Carvalho', 'São Carlos')]

## Exercício 5 — Produtos nunca vendidos

Uma solução possível:

In [27]:
sql = """
SELECT produtos.id_produto, produtos.nome, produtos.categoria, produtos.preco
FROM produtos
LEFT JOIN itens_pedido ON itens_pedido.id_produto = produtos.id_produto
WHERE itens_pedido.id_produto IS NULL
ORDER BY produtos.id_produto;
"""
cursor.execute(sql)
cursor.fetchall()

[(36, 'Mousepad', 'Periféricos', 3970.89),
 (37, 'Monitor Ultrawide', 'Monitores', 1120.07),
 (38, 'Filtro de Linha', 'Acessórios', 2107.25),
 (39, 'Headset', 'Periféricos', 2758.18),
 (40, 'Soundbar', 'Áudio', 1724.08)]

## Exercício 6 — JOIN em cadeia

Uma solução possível:

In [28]:
sql = """
SELECT clientes.nome AS cliente,
       pedidos.id_pedido,
       pedidos.data_pedido,
       produtos.nome AS produto,
       itens_pedido.quantidade,
       produtos.preco,
       ROUND(itens_pedido.quantidade * produtos.preco, 2) AS valor_item
FROM clientes
INNER JOIN pedidos ON pedidos.id_cliente = clientes.id_cliente
INNER JOIN itens_pedido ON itens_pedido.id_pedido = pedidos.id_pedido
INNER JOIN produtos ON produtos.id_produto = itens_pedido.id_produto
ORDER BY pedidos.id_pedido, produtos.id_produto
LIMIT 20;
"""
cursor.execute(sql)
cursor.fetchall()

[('Eduarda Ribeiro',
  1,
  '2026-01-15',
  'Suporte para Notebook',
  4,
  3830.81,
  15323.24),
 ('Eduarda Ribeiro', 1, '2026-01-15', 'Notebook', 2, 4324.72, 8649.44),
 ('Eduarda Ribeiro', 1, '2026-01-15', 'Controle', 2, 3734.32, 7468.64),
 ('Henrique Mendes', 2, '2026-09-11', 'Adaptador USB-C', 2, 3310.07, 6620.14),
 ('Henrique Mendes', 2, '2026-09-11', 'Mouse', 3, 3743.37, 11230.11),
 ('Henrique Mendes', 2, '2026-09-11', 'Webcam', 3, 417.38, 1252.14),
 ('Yasmin Nascimento', 3, '2026-06-16', 'SSD 500GB', 1, 3482.99, 3482.99),
 ('Yasmin Nascimento',
  3,
  '2026-06-16',
  'Suporte para Monitor',
  4,
  571.12,
  2284.48),
 ('Yasmin Nascimento', 3, '2026-06-16', 'Notebook', 2, 4324.72, 8649.44),
 ('Yasmin Nascimento', 3, '2026-06-16', 'Cabo HDMI', 2, 3191.67, 6383.34),
 ('Yasmin Nascimento', 3, '2026-06-16', 'Teclado', 3, 1977.02, 5931.06),
 ('Yasmin Pereira', 4, '2026-04-04', 'Desktop', 3, 369.7, 1109.1),
 ('Yasmin Pereira',
  4,
  '2026-04-04',
  'Suporte para Monitor',
  3,
  571.1

## Exercício 7 — Valor total de cada pedido

Uma solução possível:

In [29]:
sql = """
SELECT pedidos.id_pedido,
       ROUND(SUM(itens_pedido.quantidade * produtos.preco), 2) AS total_pedido
FROM pedidos
INNER JOIN itens_pedido ON itens_pedido.id_pedido = pedidos.id_pedido
INNER JOIN produtos ON produtos.id_produto = itens_pedido.id_produto
GROUP BY pedidos.id_pedido
ORDER BY total_pedido DESC
LIMIT 10;
"""
cursor.execute(sql)
cursor.fetchall()


[(133, 52062.53),
 (143, 51543.61),
 (173, 50088.62),
 (101, 49339.82),
 (155, 49156.87),
 (174, 47501.43),
 (41, 46073.46),
 (107, 45808.83),
 (85, 45726.23),
 (176, 44417.11)]

## Exercício 8 — Unidades vendidas por produto

Uma solução possível:

In [30]:
sql = """
SELECT produtos.id_produto,
       produtos.nome,
       COALESCE(SUM(itens_pedido.quantidade), 0) AS unidades_vendidas
FROM produtos
LEFT JOIN itens_pedido ON itens_pedido.id_produto = produtos.id_produto
GROUP BY produtos.id_produto, produtos.nome
ORDER BY produtos.id_produto;
"""
cursor.execute(sql)
cursor.fetchall()


[(1, 'Desktop', 51),
 (2, 'Mochila para Notebook', 45),
 (3, 'Roteador', 62),
 (4, 'Carregador', 56),
 (5, 'SSD 500GB', 59),
 (6, 'Suporte para Notebook', 77),
 (7, 'Adaptador Wi-Fi', 66),
 (8, 'Suporte para Monitor', 83),
 (9, 'Switch', 64),
 (10, 'Mini PC', 73),
 (11, 'Fone Bluetooth', 37),
 (12, 'SSD 1TB', 62),
 (13, 'Produto 4', 49),
 (14, 'Monitor 27', 38),
 (15, 'Cabo de Rede', 41),
 (16, 'Produto 2', 63),
 (17, 'Microfone', 59),
 (18, 'Chromebook', 71),
 (19, 'Notebook', 56),
 (20, 'Produto 1', 54),
 (21, 'Controle', 74),
 (22, 'Cabo HDMI', 45),
 (23, 'Produto 3', 43),
 (24, 'Adaptador USB-C', 73),
 (25, 'Caixa de Som', 77),
 (26, 'Pen Drive 64GB', 72),
 (27, 'Hub USB', 77),
 (28, 'DAC USB', 64),
 (29, 'Cabo USB-C', 57),
 (30, 'Teclado', 65),
 (31, 'HD Externo 1TB', 73),
 (32, 'Monitor 24', 69),
 (33, 'Monitor 4K', 67),
 (34, 'Mouse', 64),
 (35, 'Webcam', 78),
 (36, 'Mousepad', 0),
 (37, 'Monitor Ultrawide', 0),
 (38, 'Filtro de Linha', 0),
 (39, 'Headset', 0),
 (40, 'Soundbar',

## Exercício 9 — Faturamento por categoria

Uma solução possível:

In [31]:
sql = """
SELECT produtos.categoria,
       ROUND(COALESCE(SUM(itens_pedido.quantidade * produtos.preco), 0), 2) AS faturamento
FROM produtos
LEFT JOIN itens_pedido ON itens_pedido.id_produto = produtos.id_produto
GROUP BY produtos.categoria
ORDER BY faturamento DESC, produtos.categoria;
"""
cursor.execute(sql)
cursor.fetchall()

[('Periféricos', 1176104.73),
 ('Acessórios', 1019990.95),
 ('Rede', 701952.4),
 ('Armazenamento', 655891.92),
 ('Outros', 563807.72),
 ('Computadores', 554235.72),
 ('Áudio', 549030.91),
 ('Monitores', 429023.41)]

## Exercício 10 — Clientes e quantidade de pedidos

Uma solução possível:

In [32]:
sql = """
SELECT clientes.id_cliente,
       clientes.nome,
       COUNT(pedidos.id_pedido) AS total_pedidos
FROM clientes
LEFT JOIN pedidos ON pedidos.id_cliente = clientes.id_cliente
GROUP BY clientes.id_cliente, clientes.nome
ORDER BY total_pedidos DESC, clientes.id_cliente;
"""
cursor.execute(sql)
cursor.fetchall()


[(59, 'Elisa Santos', 7),
 (30, 'Carla Mendes', 6),
 (37, 'Sabrina Costa', 6),
 (10, 'Paula Martins', 5),
 (21, 'Henrique Rocha', 5),
 (35, 'Ana Carvalho', 5),
 (71, 'Giovana Mendes', 5),
 (74, 'Felipe Carvalho', 5),
 (78, 'Kaique Almeida', 5),
 (80, 'Kaique Souza', 5),
 (81, 'Igor Mendes', 5),
 (82, 'Ana Rocha', 5),
 (34, 'Marcos Gomes', 4),
 (36, 'Tiago Fernandes', 4),
 (45, 'Marcos Fernandes', 4),
 (53, 'Wesley Alves', 4),
 (57, 'Wesley Carvalho', 4),
 (64, 'Vanessa Martins', 4),
 (70, 'Daniel Martins', 4),
 (77, 'Yasmin Gomes', 4),
 (87, 'Ana Nascimento', 4),
 (89, 'Caio Martins', 4),
 (90, 'Caio Rocha', 4),
 (4, 'Yasmin Pereira', 3),
 (12, 'Natalia Costa', 3),
 (13, 'Felipe Costa', 3),
 (16, 'Julia Nascimento', 3),
 (19, 'Felipe Silva', 3),
 (22, 'Yasmin Costa', 3),
 (23, 'Rafael Costa', 3),
 (25, 'Natalia Monteiro', 3),
 (31, 'Paula Ribeiro', 3),
 (38, 'Caio Almeida', 3),
 (40, 'Bruno Souza', 3),
 (41, 'Gabriel Monteiro', 3),
 (42, 'Yasmin Rocha', 3),
 (43, 'Elisa Nascimento', 3)

## Exercício 11 — Faturamento por cidade

Uma solução possível:

In [33]:
sql = """
SELECT clientes.cidade,
       ROUND(COALESCE(SUM(itens_pedido.quantidade * produtos.preco), 0), 2) AS faturamento
FROM clientes
LEFT JOIN pedidos ON pedidos.id_cliente = clientes.id_cliente
LEFT JOIN itens_pedido ON itens_pedido.id_pedido = pedidos.id_pedido
LEFT JOIN produtos ON produtos.id_produto = itens_pedido.id_produto
GROUP BY clientes.cidade
ORDER BY faturamento DESC, clientes.cidade;
"""
cursor.execute(sql)
cursor.fetchall()

[('Bauru', 857355.33),
 ('Rio Claro', 694171.59),
 ('Piracicaba', 625730.44),
 ('Araraquara', 586775.62),
 ('São Carlos', 570839.61),
 ('Limeira', 561458.75),
 ('São Paulo', 534957.73),
 ('Ribeirão Preto', 470282.41),
 ('Campinas', 417720.49),
 ('Sorocaba', 330745.79)]

## Exercício 12 — Validação das restrições

A resolução testa uma FK inválida, a PK composta e consulta novamente a estrutura de itens_pedido.

In [ ]:
try:
    cursor.execute("""
        INSERT INTO pedidos (id_pedido, id_cliente, data_pedido)
        VALUES (999, 9999, "2026-10-01")
    """)
    conexao.commit()
except sqlite3.IntegrityError as erro:
    conexao.rollback()
    print("FK rejeitada:", erro)

item = cursor.execute(
    "SELECT id_pedido, id_produto, quantidade FROM itens_pedido LIMIT 1"
).fetchone()
try:
    cursor.execute("INSERT INTO itens_pedido VALUES (?, ?, ?)", item)
    conexao.commit()
except sqlite3.IntegrityError as erro:
    conexao.rollback()
    print("PK composta rejeitada:", erro)

for linha in cursor.execute("PRAGMA table_info(itens_pedido)"):
    print(linha)

## Encerramento

A PK composta de itens_pedido é (id_pedido, id_produto); por isso o mesmo produto não pode ser repetido no mesmo pedido. LEFT JOIN preserva registros sem correspondência, enquanto INNER JOIN retorna apenas correspondências.

In [34]:
conexao.close()
print("Conexão encerrada.")

Conexão encerrada.
